In [ ]:
# Cell 1: Install all dependencies (takes ~2-3 minutes)
!pip install -q torch transformers peft bitsandbytes datasets accelerate
!pip install -q sentence-transformers chromadb gradio sqlparse wandb
!pip install -q huggingface_hub tqdm pandas matplotlib seaborn

print("✅ All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [ ]:
# Cell 2: Create directory structure
import os

directories = [
    'data/raw',
    'data/processed',
    'data/schema_docs',
    'models',
    'results/visualizations',
    'chroma_db'
]

for d in directories:
    os.makedirs(d, exist_ok=True)
    print(f"✅ Created: {d}")

print("\n📁 Project structure ready!")

✅ Created: data/raw
✅ Created: data/processed
✅ Created: data/schema_docs
✅ Created: models
✅ Created: results/visualizations
✅ Created: chroma_db

📁 Project structure ready!


In [ ]:
# Cell 3: Login to Hugging Face
from huggingface_hub import login

login()

In [ ]:
# Cell 4: Load Spider Dataset
from datasets import load_dataset

print("📥 Loading Spider dataset...")
spider = load_dataset("spider")

print(f"\n✅ Dataset loaded!")
print(f"Training samples: {len(spider['train'])}")
print(f"Validation samples: {len(spider['validation'])}")

# Look at one example
print("\n" + "="*50)
print("SAMPLE EXAMPLE:")
print("="*50)
example = spider['train'][0]
print(f"Question: {example['question']}")
print(f"SQL: {example['query']}")
print(f"Database: {example['db_id']}")

📥 Loading Spider dataset...


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]


✅ Dataset loaded!
Training samples: 7000
Validation samples: 1034

SAMPLE EXAMPLE:
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56
Database: department_management


In [ ]:
# Cell 5: Data Preparation
import json
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ============================================================
# STEP 1: Format for Mistral Instruction Tuning
# ============================================================

def format_example(example):
    """Convert to Mistral instruction format"""

    system_prompt = """You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else."""

    prompt = f"""<s>[INST] {system_prompt}

Database: {example['db_id']}

Question: {example['question']} [/INST]
{example['query']}</s>"""

    return {
        'text': prompt,
        'question': example['question'],
        'query': example['query'],
        'db_id': example['db_id']
    }

# ============================================================
# STEP 2: Process Dataset
# ============================================================

print("🔄 Formatting training data...")
train_formatted = [format_example(ex) for ex in tqdm(spider['train'])]

print("🔄 Splitting validation into val/test...")
val_data = list(spider['validation'])
val_split, test_split = train_test_split(val_data, test_size=0.5, random_state=42)

val_formatted = [format_example(ex) for ex in tqdm(val_split)]
test_formatted = [format_example(ex) for ex in tqdm(test_split)]

# ============================================================
# STEP 3: Create Hugging Face Dataset
# ============================================================

dataset = DatasetDict({
    'train': Dataset.from_list(train_formatted),
    'validation': Dataset.from_list(val_formatted),
    'test': Dataset.from_list(test_formatted)
})

# Save to disk
dataset.save_to_disk('data/processed/sql_dataset')

print("\n✅ Dataset processed and saved!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

# ============================================================
# STEP 4: View Sample
# ============================================================

print("\n" + "="*50)
print("FORMATTED SAMPLE:")
print("="*50)
print(dataset['train'][0]['text'][:600])

🔄 Formatting training data...


100%|██████████| 7000/7000 [00:01<00:00, 4376.24it/s]


🔄 Splitting validation into val/test...


100%|██████████| 517/517 [00:00<00:00, 419268.21it/s]


Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]


✅ Dataset processed and saved!
   Train: 7000 samples
   Validation: 517 samples
   Test: 517 samples

FORMATTED SAMPLE:
<s>[INST] You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else.

Database: department_management

Question: How many heads of the departments are older than 56 ? [/INST]
SELECT count(*) FROM head WHERE age  >  56</s>


In [ ]:
# Cell 5: Data Preparation
import json
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ============================================================
# STEP 1: Format for Mistral Instruction Tuning
# ============================================================

def format_example(example):
    """Convert to Mistral instruction format"""

    system_prompt = """You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else."""

    prompt = f"""<s>[INST] {system_prompt}

Database: {example['db_id']}

Question: {example['question']} [/INST]
{example['query']}</s>"""

    return {
        'text': prompt,
        'question': example['question'],
        'query': example['query'],
        'db_id': example['db_id']
    }

# ============================================================
# STEP 2: Process Dataset
# ============================================================

print("🔄 Formatting training data...")
train_formatted = [format_example(ex) for ex in tqdm(spider['train'])]

print("🔄 Splitting validation into val/test...")
val_data = list(spider['validation'])
val_split, test_split = train_test_split(val_data, test_size=0.5, random_state=42)

val_formatted = [format_example(ex) for ex in tqdm(val_split)]
test_formatted = [format_example(ex) for ex in tqdm(test_split)]

# ============================================================
# STEP 3: Create Hugging Face Dataset
# ============================================================

dataset = DatasetDict({
    'train': Dataset.from_list(train_formatted),
    'validation': Dataset.from_list(val_formatted),
    'test': Dataset.from_list(test_formatted)
})

# Save to disk
dataset.save_to_disk('data/processed/sql_dataset')

print("\n✅ Dataset processed and saved!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

# ============================================================
# STEP 4: View Sample
# ============================================================

print("\n" + "="*50)
print("FORMATTED SAMPLE:")
print("="*50)
print(dataset['train'][0]['text'][:600])

🔄 Formatting training data...


100%|██████████| 7000/7000 [00:00<00:00, 9018.46it/s]


🔄 Splitting validation into val/test...


100%|██████████| 517/517 [00:00<00:00, 612039.28it/s]


Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/517 [00:00<?, ? examples/s]


✅ Dataset processed and saved!
   Train: 7000 samples
   Validation: 517 samples
   Test: 517 samples

FORMATTED SAMPLE:
<s>[INST] You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Only output the SQL query, nothing else.

Database: department_management

Question: How many heads of the departments are older than 56 ? [/INST]
SELECT count(*) FROM head WHERE age  >  56</s>


In [ ]:
# Cell 6: Extract Schema Docs
schema_docs = []
seen_dbs = set()

# Extract unique database schemas
for split in ['train', 'validation']:
    for example in spider[split]:
        db_id = example['db_id']
        if db_id not in seen_dbs:
            seen_dbs.add(db_id)
            schema_docs.append({
                'db_id': db_id,
                'description': f"Database schema for {db_id}"
            })

# Save schema docs
with open('data/schema_docs/schemas.json', 'w') as f:
    json.dump(schema_docs, f, indent=2)

print(f"✅ Extracted {len(schema_docs)} unique database schemas")

✅ Extracted 160 unique database schemas


In [ ]:
# Cell 7: Create Business Glossary
business_glossary = [
    {
        "term": "revenue",
        "sql_mapping": "SUM(price * quantity) or SUM(amount)",
        "context": "Total money received from sales"
    },
    {
        "term": "active users",
        "sql_mapping": "WHERE last_login > DATE_SUB(NOW(), INTERVAL 30 DAY)",
        "context": "Users who logged in within last 30 days"
    },
    {
        "term": "conversion rate",
        "sql_mapping": "COUNT(purchases) / COUNT(visits) * 100",
        "context": "Percentage of visitors who made a purchase"
    },
    {
        "term": "churn",
        "sql_mapping": "Users not active in last 90 days",
        "context": "Customers who stopped using the service"
    },
    {
        "term": "year over year",
        "sql_mapping": "Compare current period to same period last year using DATE functions",
        "context": "YoY growth comparison"
    },
    {
        "term": "top performers",
        "sql_mapping": "ORDER BY metric DESC LIMIT N",
        "context": "Highest ranked items by a metric"
    },
    {
        "term": "average order value",
        "sql_mapping": "AVG(order_total)",
        "context": "Mean value of all orders"
    },
    {
        "term": "month to date",
        "sql_mapping": "WHERE date >= DATE_TRUNC('month', CURRENT_DATE)",
        "context": "From start of current month to today"
    }
]

with open('data/schema_docs/business_glossary.json', 'w') as f:
    json.dump(business_glossary, f, indent=2)

print(f"✅ Created business glossary with {len(business_glossary)} terms")
for term in business_glossary:
    print(f"   - {term['term']}")

✅ Created business glossary with 8 terms
   - revenue
   - active users
   - conversion rate
   - churn
   - year over year
   - top performers
   - average order value
   - month to date


In [ ]:
# Cell 8: Setup RAG Pipeline
import chromadb
from chromadb.utils import embedding_functions

print("🔄 Setting up vector store...")

# Initialize embedding function
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Create collections
schema_collection = client.get_or_create_collection(
    name="schema_docs",
    embedding_function=embedding_fn
)

glossary_collection = client.get_or_create_collection(
    name="business_glossary",
    embedding_function=embedding_fn
)

print("✅ Vector store initialized!")

ModuleNotFoundError: No module named 'chromadb'

In [ ]:
# Fix: Install ChromaDB
!pip install -q chromadb

print("✅ ChromaDB installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46

In [ ]:
# Cell 8: Setup RAG Pipeline
import chromadb
from chromadb.utils import embedding_functions

print("🔄 Setting up vector store...")

# Initialize embedding function
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Create collections
schema_collection = client.get_or_create_collection(
    name="schema_docs",
    embedding_function=embedding_fn
)

glossary_collection = client.get_or_create_collection(
    name="business_glossary",
    embedding_function=embedding_fn
)

print("✅ Vector store initialized!")

🔄 Setting up vector store...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store initialized!


In [ ]:
# Cell 9: Populate Vector Store

# Load schema docs
with open('data/schema_docs/schemas.json', 'r') as f:
    schemas = json.load(f)

# Add schemas to collection
print("🔄 Adding schema documents...")
schema_documents = []
schema_ids = []
schema_metadatas = []

for i, schema in enumerate(schemas):
    doc_text = f"Database: {schema['db_id']}\nDescription: {schema['description']}"
    schema_documents.append(doc_text)
    schema_ids.append(f"schema_{i}")
    schema_metadatas.append({"db_id": schema['db_id']})

schema_collection.add(
    documents=schema_documents,
    ids=schema_ids,
    metadatas=schema_metadatas
)
print(f"✅ Added {len(schema_documents)} schema documents")

# Add glossary to collection
print("🔄 Adding glossary terms...")
with open('data/schema_docs/business_glossary.json', 'r') as f:
    glossary = json.load(f)

glossary_documents = []
glossary_ids = []
glossary_metadatas = []

for i, term in enumerate(glossary):
    doc_text = f"Term: {term['term']}\nSQL Pattern: {term['sql_mapping']}\nContext: {term['context']}"
    glossary_documents.append(doc_text)
    glossary_ids.append(f"glossary_{i}")
    glossary_metadatas.append({"term": term['term']})

glossary_collection.add(
    documents=glossary_documents,
    ids=glossary_ids,
    metadatas=glossary_metadatas
)
print(f"✅ Added {len(glossary_documents)} glossary terms")

print(f"\n📊 Vector Store Summary:")
print(f"   Schema documents: {schema_collection.count()}")
print(f"   Glossary terms: {glossary_collection.count()}")

🔄 Adding schema documents...
✅ Added 160 schema documents
🔄 Adding glossary terms...
✅ Added 8 glossary terms

📊 Vector Store Summary:
   Schema documents: 160
   Glossary terms: 8


In [ ]:
# Cell 10: RAG Retriever Class

class RAGRetriever:
    """Handles retrieval for SQL generation"""

    def __init__(self, chroma_client):
        self.client = chroma_client
        embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.schema_collection = chroma_client.get_collection("schema_docs", embedding_function=embedding_fn)
        self.glossary_collection = chroma_client.get_collection("business_glossary", embedding_function=embedding_fn)

    def retrieve_schema(self, query, top_k=3):
        """Retrieve relevant schema information"""
        results = self.schema_collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results['documents'][0] if results['documents'] else []

    def retrieve_glossary(self, query, top_k=2):
        """Retrieve relevant business terms"""
        results = self.glossary_collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results['documents'][0] if results['documents'] else []

    def get_context(self, question, db_id=None):
        """Get combined context for SQL generation"""
        schema_context = self.retrieve_schema(question)
        glossary_context = self.retrieve_glossary(question)

        context_parts = []

        if schema_context:
            context_parts.append("### Relevant Schema Information:")
            context_parts.extend(schema_context)

        if glossary_context:
            context_parts.append("\n### Relevant Business Terms:")
            context_parts.extend(glossary_context)

        return "\n".join(context_parts)

# Initialize retriever
retriever = RAGRetriever(client)
print("✅ RAG Retriever initialized!")

✅ RAG Retriever initialized!


In [ ]:
# Cell 11: Test RAG Pipeline

test_questions = [
    "Show me total revenue by month",
    "Find all active users who signed up last year",
    "What are the top 5 selling products?",
    "Calculate the average order value per customer"
]

print("="*60)
print("🧪 TESTING RAG RETRIEVAL")
print("="*60)

for question in test_questions:
    print(f"\n📝 Question: {question}")
    print("-" * 40)
    context = retriever.get_context(question)
    print(f"📚 Retrieved Context:\n{context[:300]}...")
    print()

🧪 TESTING RAG RETRIEVAL

📝 Question: Show me total revenue by month
----------------------------------------
📚 Retrieved Context:
### Relevant Schema Information:
Database: school_finance
Description: Database schema for school_finance
Database: entertainment_awards
Description: Database schema for entertainment_awards
Database: insurance_fnol
Description: Database schema for insurance_fnol

### Relevant Business Terms:
Term: ...


📝 Question: Find all active users who signed up last year
----------------------------------------
📚 Retrieved Context:
### Relevant Schema Information:
Database: party_people
Description: Database schema for party_people
Database: student_transcripts_tracking
Description: Database schema for student_transcripts_tracking
Database: museum_visit
Description: Database schema for museum_visit

### Relevant Business Terms...


📝 Question: What are the top 5 selling products?
----------------------------------------
📚 Retrieved Context:
### Relevant Schema Informa

In [ ]:
# Cell 12: Final Verification
import os

print("="*60)
print("✅ DAY 1 COMPLETION CHECKLIST")
print("="*60)

checks = [
    ("data/processed/sql_dataset", "Dataset"),
    ("data/schema_docs/schemas.json", "Schema docs"),
    ("data/schema_docs/business_glossary.json", "Business glossary"),
    ("chroma_db", "Vector store")
]

all_good = True
for path, name in checks:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {name}: {path}")
    if not exists:
        all_good = False

# Test dataset loading
from datasets import load_from_disk
dataset = load_from_disk("data/processed/sql_dataset")
print(f"\n📊 Dataset Summary:")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['validation'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

# Test retrieve

✅ DAY 1 COMPLETION CHECKLIST
✅ Dataset: data/processed/sql_dataset
✅ Schema docs: data/schema_docs/schemas.json
✅ Business glossary: data/schema_docs/business_glossary.json
✅ Vector store: chroma_db

📊 Dataset Summary:
   Train: 7000 samples
   Validation: 517 samples
   Test: 517 samples
